<a href="https://colab.research.google.com/github/owolat4real/Customer_Retention/blob/main/Stock_price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os
import torch
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.model_selection import train_test_split
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.stattools import adfuller


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(device)

In [ ]:

os.getcwd()

In [ ]:
os.listdir('/content')


In [ ]:
stock_price = pd.read_csv("/content/drive/MyDrive/Data/stock_prices.csv", index_col = [0])
secondary_store_price = pd.read_csv("/content/drive/MyDrive/Data/secondary_stock_prices.csv", index_col = 0)


In [ ]:
stock_price.head(5)

In [ ]:
secondary_store_price.head(5)

In [ ]:
df = pd.concat(
    [stock_price.assign(source="main"),
     secondary_store_price.assign(source="secondary")]
)
df.head()


# df = pd.concat(
#     [stock_price,
#      secondary_store_price]
# )
# df.head()

In [ ]:
df = df.sort_values(["SecuritiesCode","Date"]).reset_index(drop = True)
df.head(5)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.isna().sum()

In [ ]:
df["SupervisionFlag"] = df["SupervisionFlag"].astype(int)

In [ ]:
df=df.drop(columns = "ExpectedDividend" , axis = 1)

In [ ]:
df.duplicated().sum()

In [ ]:
df.dropna(inplace=True)

In [ ]:
df["Open"] = df["Open"].fillna(df["Open"].mean())
df["High"] = df["High"].fillna(df["High"].mean())
df["Low"] = df["Low"].fillna(df["Low"].mean())
df["Close"] = df["Close"].fillna(df["Close"].mean())

In [ ]:
df.isna().sum()

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data=df, x="Date", y="Open", marker="o", color="blue", linewidth=2)
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
sns.lineplot(data=df, x='Date', y='Open', marker='o', label='Open')
sns.lineplot(data=df, x='Date', y='Close', marker='x', label='Close')
plt.title('Stock Open and Close Prices Over Time')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(12,6))
sns.lineplot(data=df, x='Date', y='High', color='green', label='High')
sns.lineplot(data=df, x='Date', y='Low', color='red', label='Low')
plt.fill_between(df['Date'], df['Low'], df['High'], color='skyblue', alpha=0.3)
plt.title('High-Low Range of Stock Prices')
plt.xlabel('Date')
plt.ylabel('Price')
plt.show()

In [ ]:
numeric_cols = ['Open','High','Low','Close','Volume','AdjustmentFactor','Target']

plt.figure(figsize=(15,8))
for i, col in enumerate(numeric_cols, 1):
    plt.subplot(2, 4, i)
    sns.histplot(df[col], kde=True, bins=30)
    plt.title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation between numeric columns')
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
sns.lineplot(data=df, x='Date', y='Volume')
plt.title('Trading Volume Over Time')
plt.xlabel('Date')
plt.ylabel('Volume')
plt.show()

In [ ]:
# import seaborn as sns
# sns.countplot(data=df, x='Target')
# plt.title('Distribution of Target')
# plt.show()


In [ ]:


# Make sure Date is datetime
df['Date'] = pd.to_datetime(df['Date'])

# Set style
sns.set_style("whitegrid")

# Create a figure with subplots
fig, axes = plt.subplots(3, 2, figsize=(18, 15))
fig.suptitle('Stock Data Dashboard', fontsize=20, fontweight='bold')

# Open and Close prices over time
sns.lineplot(data=df, x='Date', y='Open', marker='o', ax=axes[0,0], label='Open')
sns.lineplot(data=df, x='Date', y='Close', marker='x', ax=axes[0,0], label='Close')
axes[0,0].set_title('Open & Close Prices Over Time')
axes[0,0].set_xlabel('Date')
axes[0,0].set_ylabel('Price')

#  High-Low price range
sns.lineplot(data=df, x='Date', y='High', color='green', ax=axes[0,1], label='High')
sns.lineplot(data=df, x='Date', y='Low', color='red', ax=axes[0,1], label='Low')
axes[0,1].fill_between(df['Date'], df['Low'], df['High'], color='lightgrey', alpha=0.3)
axes[0,1].set_title('High-Low Range Over Time')
axes[0,1].set_xlabel('Date')
axes[0,1].set_ylabel('Price')

# Histogram of numeric columns
numeric_cols = ['Open','High','Low','Close','Volume','AdjustmentFactor','Target']
sns.histplot(df['Open'], bins=30, kde=True, ax=axes[1,0], color='skyblue')
axes[1,0].set_title('Distribution of Open Prices')

sns.histplot(df['Volume'], bins=30, kde=True, ax=axes[1,1], color='salmon')
axes[1,1].set_title('Distribution of Volume')

#  Correlation heatmap (on a separate figure for clarity)
plt.figure(figsize=(10,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap', fontsize=16)
plt.show()

#  Target distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df, x='Target', palette='pastel')
plt.title('Target Distribution')
plt.show()

# Adjust layout of dashboard
plt.tight_layout()
plt.show()


In [ ]:
# checking yesteday closing prince to that of 5 days ago closing price by grouping the seciuritycode by close market
#lag 1
df["Close_lag1"] = df.groupby("SecuritiesCode")["Close"].shift(1)
#lag 5
df["Close_lag5"] = df.groupby("SecuritiesCode")["Close"].shift(5)


In [ ]:
#lag1
df["volume_lag1"] = df.groupby("SecuritiesCode")["Volume"].shift(1)
#lag1
df["volume_lag5"] = df.groupby("SecuritiesCode")["Volume"].shift(5)

In [ ]:
#lag1
df["Open_lag1"] = df.groupby("SecuritiesCode")["Open"].shift(1)
#lag1
df["Open_lag5"] = df.groupby("SecuritiesCode")["Open"].shift(5)

In [ ]:
#lag1
df["Low_lag1"] = df.groupby("SecuritiesCode")["Low"].shift(1)
#lag1
df["Low_lag5"] = df.groupby("SecuritiesCode")["Low"].shift(5)

In [ ]:
#lag1
df["High_lag1"] = df.groupby("SecuritiesCode")["High"].shift(1)
#lag1
df["High_lag5"] = df.groupby("SecuritiesCode")["High"].shift(5)

In [ ]:
# Daily return (current close vs previous close)
df["return_lag1"] = (df["Close"] - df["Close_lag1"]) / df["Close_lag1"]

# 5-day return
df["return_lag5"] = (df["Close"] - df["Close_lag5"]) / df["Close_lag5"]

In [ ]:
#Volatility is usually standard deviation of returns over a window:

# 5-day rolling volatility
df["volatility_5"] = df.groupby("SecuritiesCode")["return_lag1"].rolling(window=5).std().reset_index(level=0, drop=True)

# 10-day rolling volatility
df["volatility_10"] = df.groupby("SecuritiesCode")["return_lag1"].rolling(window=10).std().reset_index(level=0, drop=True)


In [ ]:
#Price range features

# Daily high-low range
df["range"] = df["High"] - df["Low"]

# Percentage range
df["range_pct"] = (df["High"] - df["Low"]) / df["Low"]

In [ ]:
#set date as index
df.set_index("Date", inplace = True)

In [ ]:
numeric_cols = [
    "Open", "High", "Low", "Close", "Volume", "AdjustmentFactor",
    "Close_lag1", "Close_lag5", "volume_lag1", "volume_lag5",
    "Open_lag1", "Open_lag5", "Low_lag1", "Low_lag5",
    "High_lag1", "High_lag5", "return_lag1", "return_lag5",
    "volatility_5", "volatility_10", "range", "range_pct"
    "Open", "High", "Low", "Close", "Volume", "AdjustmentFactor",    "Close_lag1", "Close_lag5", "volume_lag1", "volume_lag5",    "Open_lag1", "Open_lag5", "Low_lag1", "Low_lag5",    "High_lag1", "High_lag5", "return_lag1", "return_lag5",    "volatility_5", "volatility_10", "range", "range_pct"]

In [ ]:


# Select only columns that exist AND are numeric
numeric_cols = df.select_dtypes(include=["float64", "int64"]).columns
high_skew = df[numeric_cols].skew()
print(high_skew)


In [ ]:
df=df.drop(columns = "SecuritiesCode" , axis = 1)

In [ ]:
skewed_cols = high_skew[high_skew.abs() > 1].index.tolist()

In [ ]:
from sklearn.preprocessing import PowerTransformer

pt = PowerTransformer(method="yeo-johnson", standardize=True)
df[skewed_cols] = pt.fit_transform(df[skewed_cols])

In [ ]:
df.skew(numeric_only=True)

In [ ]:
df["SupervisionFlag"] = np.log1p(df["SupervisionFlag"])

In [ ]:
le = LabelEncoder()
df["SupervisionFlag_encoded"] = le.fit_transform(df["SupervisionFlag"])

df.drop(columns=["SupervisionFlag"], inplace=True)

In [ ]:
#use Open as the time series
ts = df["Open"]



In [ ]:
plt.figure(figsize=(10,5))
sns.lineplot(data = ts)
plt.title("Open Price Over Time")
plt.xlabel("Date")
plt.ylabel("Open Price")
plt.show()

In [ ]:
#checking Differencing cos (Differencing is needed

ts_diff = ts.diff().dropna()

plt.figure(figsize=(12,5))
sns.lineplot(data=ts_diff)
plt.title('Differenced Open Prices')
plt.show()

In [ ]:
#ARIMA requires stationary data (mean & variance don’t change over time)



result = adfuller(df['Close'].dropna())
print('ADF Statistic:', result[0])
print('p-value:', result[1])


#p-value < 0.05 → reject null → series is stationary

#p-value > 0.05 → fail to reject → series is non-stationary

In [ ]:
#We can start with ARIMA(p,d,q). You can experiment with p, d, q.



# Example: ARIMA(5,1,0) → p=5, d=1, q=0
model = ARIMA(ts, order=(5,1,0))
model_fit = model.fit()

# Summary of the model
print(model_fit.summary())


In [ ]:
# Forecast future values


# Forecast next 30 days
forecast = model_fit.forecast(steps=30)

plt.figure(figsize=(12,5))
plt.plot(ts, label='Original')
plt.plot(forecast.index, forecast, label='Forecast', color='red')
plt.title('ARIMA Forecast of Open Prices')
plt.xlabel('Date')
plt.ylabel('Open Price')
plt.legend()
plt.show()

In [ ]:
# Use SARIMA for seasonality
# If your stock has weekly/monthly seasonality, use SARIMA:



# Example: SARIMA(p,d,q)(P,D,Q,s)
sarima_model = SARIMAX(ts, order=(1,1,1), seasonal_order=(1,1,1,12))
sarima_fit = sarima_model.fit(disp=False)
print(sarima_fit.summary())

# Forecast
sarima_forecast = sarima_fit.get_forecast(steps=30).predicted_mean

plt.figure(figsize=(12,5))
plt.plot(ts, label='Original')
plt.plot(sarima_forecast.index, sarima_forecast, label='SARIMA Forecast', color='green')
plt.legend()
plt.show()

In [ ]:
df['target'] = df['Close'].shift(-1)
X = df.drop(columns=['target', 'Close'])
y = df['target']

In [ ]:


X_train,X_test,y_train,y_test = train_test_split(X,y, test_size = 0.2 , random_state = 42)



In [ ]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
print(np.isinf(X_train).sum())
print(np.isnan(X_train).sum())
print(np.isinf(y_train).sum())
print(np.isnan(y_train).sum())



In [ ]:
# Replace inf with NaN

X_train = X_train.replace([np.inf, -np.inf], np.nan)
X_test  = X_test.replace([np.inf, -np.inf], np.nan)
y_train = y_train.replace([np.inf, -np.inf], np.nan)
y_test  = y_test.replace([np.inf, -np.inf], np.nan)


In [ ]:
# Compute train statistics ONCE
X_train_mean = X_train.mean()
y_train_mean = y_train.mean()

In [ ]:
# Fill using TRAIN statistics

X_train = X_train.fillna(X_train.mean())
X_test  = X_test.fillna(X_test.mean())
y_train = y_train.fillna(y_train.mean())
y_test  = y_test.fillna(y_test.mean())


In [ ]:
print(np.isinf(X_train).sum())
print(np.isnan(X_train).sum())
print(np.isinf(y_train).sum())
print(np.isnan(y_train).sum())

In [ ]:
#XGBRegressor

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("model", XGBRegressor(
        missing=np.nan,
        n_jobs=-1,
        random_state=42
    ))
])

model = pipeline.fit(X_train, y_train)

In [ ]:

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test,y_pred)
mae = mean_absolute_error(y_test,y_pred)
r2 = r2_score(y_test,y_pred)

In [ ]:
print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)

In [ ]:
#Actual vs Predicted Plot

plt.figure()
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted")
plt.show()



In [ ]:
#Residuals Plot

residuals = y_test - y_pred

plt.figure()
plt.scatter(y_pred, residuals)
plt.axhline(y=0)
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residuals Plot")
plt.show()


In [ ]:
# LinearRegression

pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("model", LinearRegression())
])

model = pipeline.fit(X_train, y_train)

In [ ]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test , y_pred)
mae = mean_absolute_error(y_test ,y_pred)
r2 = r2_score(y_test,y_pred)

In [ ]:
print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)

In [ ]:
#Lasso Regularization model


LS = Lasso(alpha = 0.1, random_state= 42)
model = LS.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test , y_pred)
mae = mean_absolute_error(y_test , y_pred)
r2_LS = r2_score(y_test ,y_pred)


In [ ]:
print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)

In [ ]:
#Ridge Regularization model

RD = Ridge(alpha=0.1 , random_state = 42)
model = RD.fit(X_train , y_train)

In [ ]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test , y_pred)
mae = mean_absolute_error(y_test , y_pred)
r2_RD = r2_score(y_test ,y_pred)

In [ ]:
print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)

In [ ]:
y_pred_test = model.predict(X_test)
y_pred_train = model.predict(X_train)

print(y_pred_test)
print(y_pred_train)

In [ ]:
#ElasticNet Regularization model
from sklearn.linear_model import ElasticNet

ELT = ElasticNet(alpha=0.1, l1_ratio=0.05, random_state=42)
model = ELT.fit(X_train , y_train)

In [ ]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test , y_pred)
mae = mean_absolute_error(y_test , y_pred)
r2_ELT = r2_score(y_test ,y_pred)

In [ ]:
print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)


In [ ]:
#Metrics Bar Chart (MSE, RMSE, MAE, R²)

metrics = ["MSE", "MAE", "R2"]
values = [mse, mae, r2]

plt.figure()
plt.bar(metrics, values)
plt.title("Model Evaluation Metrics")
plt.show()

In [ ]:


#Compare Multiple Models

models = ["Lasso", "Ridge", "ElasticNet"]
r2_scores = [r2_LS, r2_RD, r2_ELT]

plt.figure()
plt.bar(models, r2_scores)
plt.xlabel("Models", fontsize = 14)
plt.ylabel("R2 Score")
plt.title("Model Comparison")
plt.show()


In [ ]:
# RandomForestREgressor

pipeline = Pipeline([
    ("imputer" , SimpleImputer(strategy = ("mean"))),
    ("model" , RandomForestRegressor(n_estimators = 100 , random_state = 42))
])


model = pipeline.fit(X_train , y_train)

In [ ]:
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test , y_pred)
mae = mean_absolute_error(y_test ,y_pred)
r2 = r2_score(y_test,y_pred)




In [ ]:
print("MSE:", mse)
print("MAE:", mae)
print("R2 Score:", r2)

In [ ]:
# Predict on both train and test

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)


# Evaluate scores

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

print(f"R2 on Training Set: {r2_train:.4f}")
print(f"R2 on Test Set    : {r2_test:.4f}")

In [ ]:
#Visualize Overfitting

plt.figure(figsize=(8,4))
plt.plot(y_train.values, label="Train Actual")
plt.plot(y_train_pred, label="Train Predicted")
plt.title("Train: Actual vs Predicted")
plt.legend()
plt.show()

plt.figure(figsize=(8,4))
plt.plot(y_test.values, label="Test Actual")
plt.plot(y_test_pred, label="Test Predicted")
plt.title("Test: Actual vs Predicted")
plt.legend()
plt.show()


In [ ]:
#GridSeachCV model

param_grid = {
    "n_estimators" : [100,200,300],
    "max_depth" : [10,20,30],
    "min_samples_split" : [2,5,10],
    "min_samples_leaf" : [1,2,4],
    "max_features" : ["auto","squrt", "log2"],
    "random_state" : [42],
    "n_jobs" : [-1],
    "bootstrap" : [True,False]
}

grid_search = GridSearchCV(estimator = pipeline ,param_grid = param_grid,  cv=5, scoring = "neg_mean_squared_erro", verbose = 1 )
model = grid_search.fit(X_train ,y_train)

print("Best param Founds", model.best_params_)
print("Best_score" , model.best_score_)
print("Best_estimator" , model.Best_estimator_)

In [ ]:
# GridSearchCV Heatmap
# max_depth vs n_estimators


# Extract results
results = grid_search.cv_results_

param1 = "param_max_depth"
param2 = "param_n_estimators"

depths = sorted(set(results[param1]))
estimators = sorted(set(results[param2]))

scores = results["mean_test_score"]

# Build score matrix
score_matrix = np.zeros((len(depths), len(estimators)))

for i, d in enumerate(depths):
    for j, n in enumerate(estimators):
        for k in range(len(scores)):
            if results[param1][k] == d and results[param2][k] == n:
                score_matrix[i, j] = scores[k]

# Plot heatmap
plt.figure()
plt.imshow(score_matrix)
plt.colorbar(label="Neg Mean Squared Error")
plt.xticks(range(len(estimators)), estimators)
plt.yticks(range(len(depths)), depths)
plt.xlabel("n_estimators")
plt.ylabel("max_depth")
plt.title("GridSearchCV Heatmap")
plt.show()

In [ ]:
df = pd.DataFrame(grid_search.cv_results_)
pivot = df.pivot(
    index="param_max_depth",
    columns="param_n_estimators",
    values="mean_test_score"
)

plt.figure()
sns.heatmap(pivot, annot=True, fmt=".3f")
plt.title("GridSearchCV Heatmap")
plt.show()